Eyeliner shade scatter plot comparing hue diversity and lightness range across eras.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_2
SELECT
  era,
  product_name,
  product_type,
  shade_name,
  finish,
  finish_group,
  c_pct,
  m_pct,
  y_pct,
  k_pct
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
WHERE product_type ILIKE '%Eyeliner%'
  AND shade_name IS NOT NULL
ORDER BY era, product_name, shade_name;

In [ ]:
import pandas as pd
import colorsys
from matplotlib.lines import Line2D

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

def rgb_to_hsl(r, g, b):
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return h * 360, l * 100, s * 100

df = dataframe_2.to_pandas() if hasattr(dataframe_2, 'to_pandas') else dataframe_2
df = df.copy()
df.columns = [col.lower() for col in df.columns]

df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)
df['hue'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[0])
df['lightness'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[1])

finish_markers = {
    'Matte': 's',
    'Metallic': 'D',
    'Glitter': '*',
    'Duochrome': 'P',
}

era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': '6 shades, mostly dark metallics clustered near black',
    'revival': '21 shades spanning the full hue and lightness spectrum',
}

def get_extreme_labels(era_df, n=4):
    extremes = set()
    if len(era_df) <= n * 2:
        return set(era_df['shade_name'])
    extremes.update(era_df.nsmallest(n, 'lightness')['shade_name'])
    extremes.update(era_df.nlargest(n, 'lightness')['shade_name'])
    extremes.update(era_df.nsmallest(2, 'hue')['shade_name'])
    extremes.update(era_df.nlargest(2, 'hue')['shade_name'])
    return extremes

def plot_eyeliner(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = df[df['era'] == era].copy()

        for finish, finish_df in era_df.groupby('finish'):
            colors = list(finish_df['rgb'])
            marker = finish_markers.get(finish, 'o')
            axis.scatter(
                finish_df['hue'],
                finish_df['lightness'],
                s=140,
                alpha=0.9,
                c=colors,
                edgecolors='black',
                linewidths=0.6,
                marker=marker,
                label=finish,
            )

        extremes = get_extreme_labels(era_df)
        for _, row in era_df.iterrows():
            if row['shade_name'] in extremes:
                axis.annotate(
                    row['shade_name'],
                    (row['hue'], row['lightness']),
                    textcoords='offset points',
                    xytext=(5, 4),
                    fontsize=9,
                    fontweight='normal',
                    alpha=0.85,
                )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Hue angle (color family)', fontsize=11, fontweight='normal')
        axis.set_xlim(-10, 370)
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Lightness (%)', fontsize=11, fontweight='normal')

    legend_handles = [
        Line2D([0], [0], marker=m, color='black', markerfacecolor='white',
               linestyle='', markersize=9, label=f)
        for f, m in finish_markers.items()
        if f in df['finish'].unique()
    ]
    if layout == 'horizontal':
        axes.flat[0].legend(handles=legend_handles, title='Finish',
                            loc='lower left', bbox_to_anchor=(0, 1.15),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        fig.suptitle('REVIVAL EYELINERS EXPLODE IN COLOR AND FINISH VARIETY',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        axes.flat[0].legend(handles=legend_handles, title='Finish',
                            loc='lower left', bbox_to_anchor=(0, 1.08),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('REVIVAL EYELINERS EXPLODE IN COLOR AND FINISH VARIETY',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_eyeliner('horizontal')
plot_eyeliner('vertical')